# TherapistGPT: full training run on Kaggle

Settings to pick before running: **Accelerator = GPU T4 x2** (one GPU is used) and **Internet = On**.

Add two secrets under *Add-ons > Secrets*:
- `ANTHROPIC_API_KEY` for synthetic data generation
- `HF_TOKEN` (write access) if you want to push the model to the Hugging Face Hub

Run the cells top to bottom. Total time is roughly 1 to 2 hours for 2,000 examples.

In [ ]:
import os

# Safe to re-run: only clone once, and always end up in compute/.
if not os.path.exists("/kaggle/working/TherapistGPT"):
    !git clone --depth 1 https://github.com/garyzhang1006/TherapistGPT.git /kaggle/working/TherapistGPT
%cd /kaggle/working/TherapistGPT/compute
!pip install -q -r requirements.txt
# Kaggle ships torchao 0.10.0, and peft 0.21.1 refuses to add the adapter while a torchao older than 0.16 is
# installed. Nothing here uses it, so remove it.
!pip uninstall -y -q torchao


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["ANTHROPIC_API_KEY"] = secrets.get_secret("ANTHROPIC_API_KEY")
try:
    os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
except Exception:
    print("No HF_TOKEN secret; the model will stay in /kaggle/working only.")
# Multi-GPU DataParallel does not help a 1.5B LoRA run and complicates gradient checkpointing.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## 1. Generate synthetic data
Start with `--n 20` to check quality and cost, then raise it. Reruns resume where they stopped.

In [ ]:
!python generate_synthetic.py --n 20 --out data/synthetic.jsonl
!head -c 1500 data/synthetic.jsonl

In [ ]:
!python generate_synthetic.py --n 2000 --out data/synthetic.jsonl --workers 8

## 2. Validate and split

In [ ]:
!python validate_data.py data/synthetic.jsonl --drop-invalid data/synthetic.clean.jsonl
!python split_data.py --inputs data/seed.jsonl data/synthetic.clean.jsonl

## 3. Baseline: the untrained model with the same prompt

In [ ]:
!python evaluate.py --base-only --limit 50 --report outputs/eval_base.json

## 4. Train

In [ ]:
!python train_lora.py --dry-run
!python train_lora.py --config config.yaml

## 5. Evaluate the fine-tuned model

In [ ]:
!python evaluate.py --adapter outputs/therapistgpt-lora/final --report outputs/eval_finetuned.json

## 6. Merge and publish
Replace the Hub id with your own username before running.

In [ ]:
HUB_ID = "your-hf-username/therapistgpt-1.5b"  # change this to publish

if HUB_ID.startswith("your-hf-username"):
    !python merge_and_export.py --adapter outputs/therapistgpt-lora/final
    print("Merged locally only. Set HUB_ID above to also push to the Hugging Face Hub.")
else:
    !python merge_and_export.py --adapter outputs/therapistgpt-lora/final --push {HUB_ID}


## 7. Try it

In [ ]:
import sys, json
sys.path.insert(0, ".")
from therapistgpt.inference import Organizer

org = Organizer("outputs/therapistgpt-merged")
print(json.dumps(org.organize("cant sleep. work tomorrow and i didnt finish the slides. my sister is mad at me and i dont even know why. im so useless"), indent=2))